# Creating Ollama instance on Kaggle Notebook:

**Pre-requisites:**  
- Kaggle account (sign up for one at https://www.kaggle.com/)
- Kaggle Notebook (create one from https://www.kaggle.com/code)

## Step 1: Download Pre-requisites and ollama
If you want to use GPU *(recommended)* on your Kaggle Notebook:  
Go to **Settings** > **Accecelerator** and pick any GPU! 😊

In [1]:
# Only need to run on first run of the container

# Download pre-requisites
!sudo apt update
!sudo apt install pciutils lshw
!sudo apt-get install zstd

# Download and install ollama to the system
!curl -fsSL https://ollama.com/install.sh | sh

# Install required python libraries
!pip install -qq ollama

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,436 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,855 kB] 
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18

In [2]:
!which ollama        # check if it exists

/usr/local/bin/ollama


## Step 2: Start the ollama server

In [3]:
# Only need to run on first run of the container

import subprocess

def start_ollama_server() -> None:
    """Starts the Ollama server."""
    subprocess.Popen(['ollama', 'serve'])
    print("Ollama server started.")

start_ollama_server()

Ollama server started.


## Step 3: Now let's download your model from ollama¶

In [4]:
# Only need to run on first run of the container

import ollama

# Setup an Ollama client with the localhost
client = ollama.Client(host='http://localhost:11434')

# Pre-load the model on the server example 'qwen3.5:9b'
client.pull(model='qwen3.5:9b') # can pull repeat this code in new cell to pull more models that you want

Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIMpYovzlqmSnIbhZBPsHHnuHcaQrU2Lu7r432qxMHZlC



time=2026-03-15T05:40:27.845Z level=INFO source=routes.go:1727 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAMA_RE

## Step 4: Let's run a test query from the model you downloaded earlier¶

In [5]:
# client.pull(model='qwen3.5:35b')

time=2026-03-15T05:40:29.138Z level=INFO source=runner.go:106 msg="experimental Vulkan support disabled.  To enable, set OLLAMA_VULKAN=1"
time=2026-03-15T05:40:29.139Z level=INFO source=server.go:430 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 33851"
time=2026-03-15T05:40:29.139Z level=INFO source=server.go:430 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 43213"
time=2026-03-15T05:40:29.139Z level=INFO source=server.go:430 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 40563"
time=2026-03-15T05:40:29.143Z level=INFO source=server.go:430 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 40223"
time=2026-03-15T05:40:29.741Z level=INFO source=types.go:42 msg="inference compute" id=GPU-17609847-b054-6c61-f43d-e4abd8f27fcf filter_id="" library=CUDA compute=7.5 name=CUDA0 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:04.0 type=discrete 

[GIN] 2026/03/15 - 05:43:42 | 200 |         3m12s |       127.0.0.1 | POST     "/api/pull"


ProgressResponse(status='success', completed=None, total=None, digest=None)

In [7]:
# be prepared to wait upon first run

def query_ollama_with_client(client: ollama.Client, prompt: str, model_id: str, img_path: str = None) -> None:
    """Queries the Ollama server using the `ollama-python` client library."""
    try:
        messages=[
            {
                'role': 'user',
                'content': prompt,
            },
        ]

        if img_path:
          messages[0]['images'] = [img_path]

        stream = client.chat(
            model=model_id,
            messages=messages,
            stream=True 
        )
        print("---Response from ollama client---")
        for chunk in stream:
            print(chunk.message.content, end='', flush=True)
        print("---End of Response---")
        print()
    except Exception as e:
        print(f"Error querying Ollama with client: {e}")

# Ask a question
query_ollama_with_client(
    client,
    "Why is the sky blue?", # change your prompt here
    "qwen3.5:9b", # using qwen3.5:9b model earlier but kaggle uses 2x T4 GPUs (can go for qwen2.5:32b as well but the wait upon first run is very long)
)

---Response from ollama client---
The sky is not green because of a phenomenon called **Rayleigh scattering** and how the human eye perceives light.

Here is the breakdown of why blue wins and green loses:

**1. Sunlight is a mix of all colors**
White sunlight contains all the colors of the rainbow (red, orange, yellow, green, blue, indigo, violet). Each color has a different wavelength.
*   **Red:** Long wavelength.
*   **Green:** Medium wavelength.
*   **Blue/Violet:** Short wavelength.

**2. The atmosphere acts like a filter**
The Earth's atmosphere is full of nitrogen and oxygen molecules. When sunlight hits these molecules, it bounces off them in different directions. This is called scattering.
*   **Shorter wavelengths scatter much more easily** than longer wavelengths.
*   Blue and violet light have the shortest wavelengths of the visible spectrum, so they scatter in all directions much more effectively than green or red light.

**3. Why blue beats green**
Because blue light has